# Notebook 24 — Chiusura del corso: sicurezza, checklist e autovalutazione finale

**Modulo:** 24 — Allineamento e generazione multimodale (ultimo modulo)
**Tempo stimato:** 1-1,5 ore

## Cosa costruirai

Questo non è un notebook tecnico come i precedenti: è la parte pratica della chiusura del
corso. Al termine avrai:
- Una checklist di sicurezza applicabile a un progetto reale, con una funzione che ne calcola
  il livello di completamento per categoria
- Un esercizio di *code review*: individuare e correggere una chiave API scritta direttamente
  nel codice (*hardcoded*) in uno snippet di esempio
- Uno strumento minimo di controllo automatico per segreti hardcoded, riutilizzabile su codice
  vero
- Una mappa di autovalutazione della tua confidenza sui sei blocchi tematici del corso
- L'**autovalutazione finale del corso** (non della settimana: è l'ultima cella dei 24 moduli)

## Prerequisiti

```bash
pip install python-dotenv
```

Non servono chiavi API reali per eseguire questo notebook: gli esempi di codice che chiamano
un LLM sono illustrativi (l'obiettivo è la struttura del codice, non il risultato della
chiamata). Se vuoi comunque il pattern completo:
```
OPENAI_API_KEY=sk-...
```

---
## Cella 0 — Setup

Lo stesso pattern usato in ogni notebook del corso, dal Modulo 01 in poi (dispensa, Sezione
3.2): chiavi caricate da `.env`, mai scritte nel codice.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_KEY = os.getenv("OPENAI_API_KEY")

def verifica_chiave(nome: str, valore: str | None) -> None:
    if valore:
        print(f"OK {nome}: {valore[:8]}...")
    else:
        print(f"NON IMPOSTATA {nome} (va bene: questo notebook non fa chiamate API reali)")

verifica_chiave("OPENAI_API_KEY", OPENAI_KEY)


---
## Sezione 1 — Checklist pratica di sicurezza

Una checklist da applicare, voce per voce, prima di portare un progetto basato su LLM davanti a
utenti reali. Le voci sono prese direttamente dalle Sezioni 2 e 3 della dispensa: rischi pratici
(prompt injection, data leakage, allucinazioni, bias, dipendenza da un singolo provider) e buone
pratiche minime (validazione dell'output, gestione delle chiavi, privacy nell'esecuzione
locale, osservabilità).

In [ ]:
CHECKLIST_SICUREZZA = [
    {"categoria": "Prompt injection", "voce": "Il testo proveniente da fonti esterne (email, pagine web, commenti) è delimitato chiaramente nel prompt (Modulo 02)", "fatto": False},
    {"categoria": "Prompt injection", "voce": "Le azioni eseguite da un agente in risposta a input non fidato hanno permessi minimi e richiedono conferma per operazioni irreversibili", "fatto": False},
    {"categoria": "Data leakage", "voce": "Nessun dato riservato (chiavi, dati personali, codice proprietario) viene incollato in prompt inviati a servizi di terze parti senza aver letto i termini di servizio", "fatto": False},
    {"categoria": "Data leakage", "voce": "Il system prompt e il contesto non possono trapelare nella risposta verso utenti che non dovrebbero vederli (specialmente in sistemi multi-tenant)", "fatto": False},
    {"categoria": "Allucinazioni", "voce": "Gli output che alimentano affermazioni fattuali verificabili (citazioni, dati numerici, riferimenti) sono sottoposti a verifica, non solo generati e pubblicati", "fatto": False},
    {"categoria": "Bias", "voce": "È stato fatto un controllo, anche informale, su come il sistema si comporta con input che variano solo per dettagli demografici", "fatto": False},
    {"categoria": "Dipendenza da un provider", "voce": "Il codice usa un wrapper multi-provider (come quello del Modulo 08) invece di chiamate dirette sparse ovunque nel codice", "fatto": False},
    {"categoria": "Validazione output", "voce": "Ogni parsing di output strutturato (JSON) è dentro un blocco try/except che gestisce il fallimento, non lo assume andato a buon fine", "fatto": False},
    {"categoria": "Validazione output", "voce": "Lo schema dell'output (tipi di campo, valori ammessi) viene validato, non solo la sintassi JSON", "fatto": False},
    {"categoria": "Gestione chiavi", "voce": "Nessuna chiave API è scritta direttamente nel codice: solo variabili d'ambiente caricate con python-dotenv", "fatto": False},
    {"categoria": "Gestione chiavi", "voce": "Il file .env è in .gitignore ed esiste un .env.example con soli placeholder", "fatto": False},
    {"categoria": "Privacy locale", "voce": "Per dati che non possono lasciare l'infrastruttura, è stata valutata l'esecuzione locale di un modello open-weight (Modulo 04)", "fatto": False},
    {"categoria": "Osservabilità", "voce": "Prompt, risposte, token e costo di ogni chiamata vengono registrati, almeno in forma minima (Modulo 08, funzione calcola_costo)", "fatto": False},
]

print(f"Voci nella checklist: {len(CHECKLIST_SICUREZZA)}")


In [ ]:
def valuta_checklist(checklist: list[dict]) -> dict:
    """Calcola la percentuale di voci completate per categoria."""
    per_categoria: dict[str, list[bool]] = {}
    for voce in checklist:
        per_categoria.setdefault(voce["categoria"], []).append(voce["fatto"])

    esito = {}
    for categoria, valori in per_categoria.items():
        percentuale = 100 * sum(valori) / len(valori)
        esito[categoria] = percentuale

    return esito


def stampa_report_checklist(checklist: list[dict]) -> None:
    esito = valuta_checklist(checklist)
    totale_fatto = sum(1 for v in checklist if v["fatto"])
    print(f"Completamento totale: {totale_fatto}/{len(checklist)} voci\n")
    for categoria, percentuale in esito.items():
        barra = "#" * round(percentuale / 10) + "-" * (10 - round(percentuale / 10))
        print(f"  {categoria:25s} [{barra}] {percentuale:5.0f}%")


stampa_report_checklist(CHECKLIST_SICUREZZA)


**TODO:** pensa a un progetto reale (tuo, del tuo lavoro, o anche solo ipotetico) e modifica
i valori `"fatto"` nella lista `CHECKLIST_SICUREZZA` sopra per riflettere lo stato reale di
quel progetto. Poi rilancia `stampa_report_checklist(CHECKLIST_SICUREZZA)` nella cella
seguente.

In [ ]:
# TODO: modifica alcuni valori "fatto" in CHECKLIST_SICUREZZA (cella sopra) pensando
# a un progetto reale o ipotetico, poi rilancia il report qui sotto

stampa_report_checklist(CHECKLIST_SICUREZZA)


---
## Sezione 2 — Esercizio di code review: trova la chiave hardcoded

Lo snippet qui sotto è un esempio di codice **da correggere**, non da eseguire così com'è. È
del tipo che capita di trovare in un repository reale quando qualcuno, sotto scadenza, scrive
"funziona, sistemo dopo". Contiene **almeno due problemi di sicurezza** legati alla gestione
delle chiavi (Sezione 3.2 della dispensa). Trovali prima di leggere la soluzione più sotto.

```python
# snippet_da_correggere.py
import openai

# la chiave qui sotto e' un placeholder di esempio, non una chiave reale
openai.api_key = "<LA-TUA-CHIAVE>"

def genera_risposta(prompt):
    risposta = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}]
    )
    print(f"Chiave usata per la chiamata: {openai.api_key}")
    return risposta.choices[0].message.content
```

**TODO:** scrivi qui sotto, in una frase per ciascuno, i problemi che hai trovato (ce ne sono
almeno due, di gravità diversa).

**I problemi che ho trovato** (modifica questa cella):

1. ...
2. ...

In [ ]:
# TODO: riscrivi lo snippet corretto qui sotto, applicando il pattern .env + python-dotenv
# (lo stesso della Cella 0 di questo notebook, e di ogni notebook del corso dal Modulo 01)

import os
from dotenv import load_dotenv
import openai

load_dotenv()

# TODO: carica la chiave da variabile d'ambiente invece che come stringa letterale
openai.api_key = None  # sostituisci con os.getenv("OPENAI_API_KEY")


def genera_risposta_corretta(prompt):
    if not openai.api_key:
        raise RuntimeError("OPENAI_API_KEY non trovata. Controlla il tuo file .env")

    risposta = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
    )
    # TODO: rimuovi qualunque stampa/log che esponga la chiave, anche parzialmente
    return risposta.choices[0].message.content


### Soluzione (confrontala con la tua)

I due problemi principali:

1. **Chiave hardcoded**: `openai.api_key = "sk-proj-..."` scritta direttamente nel codice.
   Se questo file finisce in un repository (anche privato, anche per errore), la chiave è
   compromessa e va revocata immediatamente — non basta cancellarla in un commit successivo,
   perché resta nella cronologia di git.
2. **Chiave stampata nei log**: `print(f"Chiave usata per la chiamata: {openai.api_key}")`
   espone la chiave in chiaro in qualunque output/log della funzione, anche se la chiave stessa
   fosse caricata correttamente da `.env`. È un problema indipendente dal primo: si può avere
   una gestione delle chiavi corretta e comunque bucarla per una `print` di troppo lasciata da
   un debug.

Versione corretta:

```python
import os
from dotenv import load_dotenv
import openai

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")


def genera_risposta_corretta(prompt):
    if not openai.api_key:
        raise RuntimeError("OPENAI_API_KEY non trovata. Controlla il tuo file .env")

    risposta = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
    )
    # non stampiamo mai la chiave, nemmeno parzialmente, nei log applicativi
    return risposta.choices[0].message.content
```

---
## Sezione 3 — Riflessione sul percorso: 24 moduli in sei blocchi

Prima dell'autovalutazione finale, un momento di riflessione più strutturato. La dispensa
(Sezione 1) riassume il percorso in sei blocchi tematici. Assegna a ciascuno un livello di
confidenza da 1 (poco sicuro, dovrei ripassare) a 5 (molto sicuro, lo spiegherei a un
collega).

In [ ]:
BLOCCHI_TEMATICI = {
    "A - Fondamenti (Moduli 01-04)": None,
    "B - Dall'interfaccia al codice (Moduli 05-08)": None,
    "C - Costruire con le API (Moduli 09-12)": None,
    "D - Come funzionano davvero gli LLM (Moduli 13-16)": None,
    "E - Specializzare un modello (Moduli 17-20)": None,
    "F - Allineamento e generazione multimodale (Moduli 21-24)": None,
}

# TODO: assegna un intero da 1 a 5 a ciascun blocco, sostituendo i None
BLOCCHI_TEMATICI["A - Fondamenti (Moduli 01-04)"] = None
BLOCCHI_TEMATICI["B - Dall'interfaccia al codice (Moduli 05-08)"] = None
BLOCCHI_TEMATICI["C - Costruire con le API (Moduli 09-12)"] = None
BLOCCHI_TEMATICI["D - Come funzionano davvero gli LLM (Moduli 13-16)"] = None
BLOCCHI_TEMATICI["E - Specializzare un modello (Moduli 17-20)"] = None
BLOCCHI_TEMATICI["F - Allineamento e generazione multimodale (Moduli 21-24)"] = None


def mostra_mappa_confidenza(blocchi: dict) -> None:
    for nome, livello in blocchi.items():
        if livello is None:
            print(f"{nome:52s} | (non ancora valutato)")
            continue
        barra = "#" * livello + "-" * (5 - livello)
        print(f"{nome:52s} | {barra} ({livello}/5)")


mostra_mappa_confidenza(BLOCCHI_TEMATICI)


**Riflessione libera** (modifica questa cella):

- Il blocco dove mi sento più solido è: ... perché ...
- Il blocco che rivedrei per primo tra un mese è: ... perché ...
- Una cosa che pensavo di sapere sull'IA generativa prima del corso, e che ho scoperto essere
  sbagliata o incompleta: ...

---
## Sfida finale — Uno strumento di audit riutilizzabile

Combina la checklist (Sezione 1) e il principio del controllo sui segreti hardcoded (Sezione 2)
in un unico strumento minimo, il tipo di controllo automatico che potresti effettivamente
integrare in un progetto reale (senza sostituire strumenti professionali di secret scanning
come `gitleaks` o `truffleHog` — il principio, però, è lo stesso: cercare pattern sospetti
prima che finiscano in produzione).

Completa i `# TODO`.

In [ ]:
import re

# Pattern euristico: rileva assegnazioni tipo `api_key = "..."` o `secret = '...'` con una
# stringa abbastanza lunga da sembrare una credenziale reale, non un placeholder vuoto
PATTERN_CHIAVE_HARDCODED = re.compile(
    r'(api[_-]?key|secret|token|password)\s*=\s*["\'][^"\']{10,}["\']',
    re.IGNORECASE,
)


def controlla_chiavi_hardcoded(codice: str) -> list[str]:
    """Scansiona una stringa di codice sorgente riga per riga e segnala quelle che
    sembrano contenere un segreto scritto direttamente nel codice."""
    problemi = []
    for numero_riga, riga in enumerate(codice.splitlines(), start=1):
        if PATTERN_CHIAVE_HARDCODED.search(riga):
            problemi.append(f"Riga {numero_riga}: possibile segreto hardcoded -> {riga.strip()}")
    return problemi


SNIPPET_DA_CORREGGERE = '''import openai

openai.api_key = "<LA-TUA-CHIAVE>"

def genera_risposta(prompt):
    risposta = openai.ChatCompletion.create(model="gpt-4", messages=[{"role": "user", "content": prompt}])
    print(f"Chiave usata per la chiamata: {openai.api_key}")
    return risposta.choices[0].message.content
'''


def audit_progetto(nome_progetto: str, checklist: list[dict], codice_da_verificare: str) -> dict:
    """Sfida finale: unisce la checklist di sicurezza (Sezione 1) e il controllo
    automatico di segreti hardcoded (Sezione 2) in un unico report di audit."""
    # TODO: calcola l'esito della checklist con valuta_checklist (Sezione 1)
    esito_checklist = {}

    # TODO: calcola i problemi nel codice con controlla_chiavi_hardcoded (definita sopra)
    problemi_codice = []

    print(f"=== Audit di sicurezza: {nome_progetto} ===\n")
    print("Checklist per categoria:")
    for categoria, percentuale in esito_checklist.items():
        print(f"  {categoria}: {percentuale:.0f}%")

    print("\nControllo segreti hardcoded nel codice fornito:")
    if problemi_codice:
        for p in problemi_codice:
            print(f"  ATTENZIONE - {p}")
    else:
        print("  Nessun problema rilevato (o il controllo non e' ancora stato completato: vedi i TODO sopra)")

    return {"checklist": esito_checklist, "problemi_codice": problemi_codice}


# Prova la tua implementazione (decommenta dopo aver completato i TODO sopra)
# audit_progetto("Il mio progetto", CHECKLIST_SICUREZZA, SNIPPET_DA_CORREGGERE)


---
## Autovalutazione finale del corso

Non è l'autovalutazione di una settimana: è l'ultima cella dei 24 moduli. Prenditi il tempo che
serve.

1. **Guardando indietro ai 24 moduli, cosa sai fare oggi che non sapevi fare all'inizio?**
   ...

2. **Quale modulo rivedresti per primo se dovessi ripassare qualcosa tra un mese?**
   ...

3. **Qual è il primo progetto reale — lavorativo o personale — in cui applicherai qualcosa di
   questo corso?**
   ...

4. **Guardando la roadmap (Sezione 6 della dispensa 24: RAG, agenti, LLMOps, sicurezza
   applicativa, specializzazione verticale), quale area approfondirai per prima? Qual è il tuo
   primo passo concreto, questa settimana e non "prima o poi"?**
   ...

5. **Tempo totale stimato dedicato all'intero corso:**
   ...

---
*Notebook 24 — Modulo 24 (ultimo) del Corso Completo di IA Generativa*